In [1]:
import gamgee

/opt/anaconda3/envs/newsam/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from multiprocessing import shared_memory
import numpy as np
import tifffile as tiff
from pathlib import Path
import subprocess


In [3]:
img = tiff.imread('test_data/20250801_experimentA_1/Granules/2024-01-21_SD2_63X_bin1_gra-GFP_E122_MO12_10hpf-1.tif')

In [4]:

shm = shared_memory.SharedMemory(create=True, size=img.nbytes)
shared_array = np.ndarray(img.shape, dtype=img.dtype, buffer=shm.buf)
shared_array[:] = img[:]

In [5]:
identifier = f"{shm.name}-{img.shape[0]}_{img.shape[1]}-{str(img.dtype)}"

In [13]:
print(f"Input identifier: {identifier}")

Input identifier: psm_4ccfb39a-366_366-uint16


In [6]:
# two dir up and then into gamgee
path_denoisepy = Path(gamgee.__file__).parent / 'caredenoising.py'

In [7]:
path_denoisepy

PosixPath('/Users/julian/Documents/General Science/Programming/py/general_analysis/20250722_msam_granule_sizes/gamgee/caredenoising.py')

In [8]:
cmd = f'conda run -n csbdeep python "{path_denoisepy}" {identifier}'
result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

In [9]:
result.stdout

"Loading network weights from 'weights_best.h5'.\nProcessing shared memory segment: psm_4ccfb39a with shape (366, 366) and dtype uint16\nRESULT_IDENTIFIER:psm_41334017-366_366-uint16\nFinished processing psm_4ccfb39a-366_366-uint16\n\n"

In [10]:
# Parse the result identifier (same format as input)
result_identifier = None
for line in result.stdout.split('\n'):
    if line.startswith('RESULT_IDENTIFIER:'):
        result_identifier = line.split(':', 1)[1]
        break

In [11]:
from gamgee.utils.denoising import get_shared_memory_info

In [12]:
if result_identifier:
    print(f"Result identifier: {result_identifier}")

    # Parse the result identifier using your existing function
    result_shm_info = get_shared_memory_info(result_identifier)

    # Access the result shared memory
    result_shm = shared_memory.SharedMemory(name=result_shm_info['name'])
    result_array = np.ndarray(result_shm_info['shape'], dtype=result_shm_info['dtype'], buffer=result_shm.buf)
    restored = result_array.copy()

    print(f"Successfully retrieved denoised image: {restored.shape}, {restored.dtype}")

    # Cleanup
    result_shm.close()
    result_shm.unlink()

    # Also cleanup original shared memory
    original_shm = shared_memory.SharedMemory(name=shm.name)
    original_shm.close()
    original_shm.unlink()
else:
    print("Could not find result identifier")

Result identifier: psm_41334017-366_366-uint16


FileNotFoundError: [Errno 2] No such file or directory: '/psm_41334017'